# ID2222 Homework 2: Discovery of Frequent Itemsets and Association Rules

**Authors:**  group 28

This notebook contains both the source code and the report for the homework. It implements:

1. The **A-Priori** algorithm to find all frequent itemsets with support at least `s`.
2. The generation of **association rules** `X -> Y` with confidence at least `c` from those itemsets.

## How to build and run

**Requirements:** Python 3 (tested with Python 3.11) and Jupyter (or VS Code with the Jupyter extension). The algorithm only uses the Python standard library; `pandas` is used only to display the association rules as a table. Install the dependencies with:

```
pip install -r requirements.txt
```

**Dataset:** T10I4D100K from the FIMI repository: <https://fimi.uantwerpen.be/data/T10I4D100K.dat> (100,000 baskets, about 4 MB). It is **not** included in the submission. The first code cell downloads it automatically to `./dataset.dat` if the file does not exist. It can also be downloaded manually from the link above and saved as `dataset.dat` next to this notebook.

**Run:**

- Open `apriori.ipynb` in Jupyter or VS Code and select *Run All*, or
- from the command line:
  ```
  pip install notebook
  jupyter nbconvert --to notebook --execute --inplace apriori.ipynb
  ```

**Parameters** (set in the *Parameters* cell below):

| Parameter | Meaning | Default |
|---|---|---|
| `s` | Minimum support, as a percentage of the baskets. `s = 1` means the itemset must appear in at least 1% of the baskets (1,000 of 100,000). | `1` |
| `c` | Minimum confidence of an association rule, between 0 and 1. | `0.5` |

## Solution

**Reading the data.** Each line of the dataset is a basket: a list of item ids separated by spaces. The file is read into `baskets`, a list of lists of integers.

**A-Priori** (class `Apriori`). The algorithm relies on monotonicity: if an itemset is frequent, all its subsets are frequent too. So frequent itemsets of size `k` can only be built from frequent itemsets of size `k-1`.

- `freq_items(s)`: **first pass**. Counts how many baskets contain each single item and keeps the items whose count is at least the threshold `s/100 * number of baskets`.
- `freq_itemsets(prev_freq, k, s)`: **pass k**. For each basket it removes the items that are not in any frequent `(k-1)`-itemset, forms all `k`-itemsets with the remaining items and counts an itemset only if all its `(k-1)`-subsets are frequent (the candidates). It then keeps the candidates that reach the threshold. Itemsets are stored as sorted tuples, so `(39, 704)` and `(704, 39)` are the same key.
- `run()`: calls `freq_items` and then `freq_itemsets` for `k = 2, 3, ...` until no frequent itemsets of size `k` are found. Returns a dictionary `{itemset: support count}` with the frequent itemsets of every size.

**Association rules** (`rules(frequent, c)`). For each frequent itemset `I` and each non-empty proper subset `X` of `I`, it considers the rule `X -> Y` with `Y = I \ X`:

`confidence(X -> Y) = support(I) / support(X)`

and keeps it if `confidence >= c`. `support(X)` is always available in `frequent` because every subset of a frequent itemset is also frequent, so the rules are computed without reading the baskets again.

In [19]:
from itertools import combinations
import os
import time
import urllib.request

#use for testing 
baskets = [
    [25, 52, 164, 240, 274, 328, 368, 448, 538, 561, 630, 687, 730, 775, 825, 834],
    [39, 120, 124, 205, 401, 581, 704, 814, 825, 834],
    [35, 249, 674, 712, 733, 759, 854, 950],
    [39, 422, 449, 704, 825, 857, 895, 937, 954, 964],
    [15, 229, 262, 283, 294, 352, 381, 708, 738, 766, 853, 883, 966, 978],
    [26, 104, 143, 320, 569, 620, 798],
    [7, 185, 214, 350, 529, 658, 682, 782, 809, 849, 883, 947, 970, 979],
    [227, 390],
    [71, 192, 208, 272, 279, 280, 300, 333, 496, 529, 530, 597, 618, 674, 675, 720, 855, 914, 932],
]

file_path ='./dataset.dat'
dataset_url = 'https://fimi.uantwerpen.be/data/T10I4D100K.dat'

#download the dataset if it is not already there
if not os.path.exists(file_path):
    urllib.request.urlretrieve(dataset_url, file_path)

baskets = []

with open(file_path, 'r') as file:
    content = file.readlines()
    for line in content:
        basket = [int(x) for x in line.strip().split(" ")] #strip to remove /n and split on " "
        baskets.extend([basket])

print('Number of baskets:', len(baskets))

class Apriori():
    def __init__(self,baskets,s):
        self.baskets = baskets
        self.count = {}
        self.s = s   
    
    def freq_items(self,s):
        self.count = {}   #reset, otherwise calling it twice doubles the counts
        for basket in self.baskets:
            for item in basket:
                if item not in self.count:
                    self.count[item] = 1
                else:
                    self.count[item] = self.count[item] + 1

        support_threshold = (s/100) * len(self.baskets)
        d = {k:v for k,v in self.count.items() if v >= support_threshold}
        return d

    ##TAKE EACH BASKET , REMOVE THE ITEMS NOT IN THE FREQUENT SET , 
    ## FOR THE REMAINING ITEMS , FORM THEM INTO SETS OF SIZE k AND IF ALL ITS (k-1)-SUBSETS ARE FREQUENT ADD ONE
    ## (for k = 2 these are the pairs; itemsets are sorted tuples so (39, 704) and (704, 39) are the same)
    ##
    ## There are two filters:
    ##  1. remaining (coarse filter, per item): drops the items that are not in ANY frequent (k-1)-itemset.
    ##     It only makes the baskets shorter, so combinations() generates fewer groups.
    ##  2. all(...) (fine filter, per group): an itemset of size k is only counted if EVERY one of its
    ##     (k-1)-subsets is frequent (monotonicity: if a subset is not frequent, the itemset cannot be frequent).
    ##     For k = 2 filter 1 already guarantees this, but from k = 3 on it does not. Example: if (1,2) and (1,3)
    ##     are frequent but (2,3) is not, items 1, 2, 3 all pass filter 1, but the triple (1,2,3) is rejected
    ##     by filter 2 because its subset (2,3) is not frequent.
    def freq_itemsets(self,prev_freq,k,s):
        freq_set = set(item for itemset in prev_freq for item in itemset) #all the products who are still frequent
        count = {}
        for basket in self.baskets:
            remaining = sorted(item for item in basket if item in freq_set)  #filter 1: it removes the non-frequent items from the basket
            for itemset in combinations(remaining, k):   #combines the remaining items in groups of k items
                #filter 2: count the group only if all its subgroups of size k-1 are frequent
                #e.g. for (1,2,3) it checks (1,2), (1,3) and (2,3)
                if all(sub in prev_freq for sub in combinations(itemset, k-1)):   
                    if itemset not in count:
                        count[itemset] = 1
                    else:
                        count[itemset] = count[itemset] + 1

        threshold = (s/100) * len(self.baskets)
        d = {itemset:v for itemset,v in count.items() if v >= threshold}
        return d

    #repeat for k = 2, 3, ... until there are no frequent itemsets of size k
    #It returns all_freq, which has the frequent itemsets from all sizes (1, 2, 3,...)
    def run(self):
        prev_freq = {(item,):v for item,v in self.freq_items(self.s).items()}
        all_freq = dict(prev_freq) #stores itemsets size 1 
        k = 2
        while prev_freq:   
            prev_freq = self.freq_itemsets(prev_freq, k, self.s)
            all_freq.update(prev_freq)
            k = k + 1
        return all_freq

    #rule X -> Y, where X and Y split a frequent itemset: confidence = support(X u Y) / support(X)
    def rules(self,frequent,c):
        rules = []
        for itemset, support in frequent.items():
            for r in range(1, len(itemset)):
                for X in combinations(itemset, r):
                    Y = tuple(item for item in itemset if item not in X)
                    confidence = support / frequent[X]   #X is frequent because it is a subset of a frequent itemset
                    if confidence >= c:
                        rules.append((X, Y, support, confidence))
        return rules

Number of baskets: 100000


## Parameters

In [20]:
s = 1   #minimum support, in % of the baskets
c = 0.5   #minimum confidence

## Frequent itemsets

In [21]:
#print an itemset as a set, e.g. (39, 704) -> {39, 704} and (39,) -> {39}
def fmt(itemset):
    return '{' + ', '.join(str(item) for item in itemset) + '}'

start = time.perf_counter()
a = Apriori(baskets,s)
frequent = a.run()
print(f'A-Priori with s = {s}%: {time.perf_counter() - start:.2f} s\n')

for k in range(1, max(len(itemset) for itemset in frequent) + 1):
    print(f'Frequent itemsets of size {k}:', sum(1 for itemset in frequent if len(itemset) == k))

print('\nFrequent itemsets with more than one item (itemset, support count):')
for itemset, v in frequent.items():
    if len(itemset) > 1:
        print(fmt(itemset), v)

A-Priori with s = 1%: 2.60 s

Frequent itemsets of size 1: 375
Frequent itemsets of size 2: 9
Frequent itemsets of size 3: 1

Frequent itemsets with more than one item (itemset, support count):
{39, 704} 1107
{39, 825} 1187
{704, 825} 1102
{227, 390} 1049
{789, 829} 1194
{368, 829} 1194
{217, 346} 1336
{368, 682} 1193
{390, 722} 1042
{39, 704, 825} 1035


## Association rules

In [22]:
import pandas as pd

start = time.perf_counter()
rules = a.rules(frequent, c)
print(f'Association rules with c = {c}: {(time.perf_counter() - start) * 1000:.3f} ms')
print(f'{len(rules)} rules found')

#show the rules as a table, sorted from highest to lowest confidence
rules_table = pd.DataFrame(
    [(fmt(X), fmt(Y), support, round(confidence, 3)) for X, Y, support, confidence in rules],
    columns=['X', 'Y', 'Support of X u Y', 'Confidence'],
)
rules_table = rules_table.sort_values('Confidence', ascending=False).reset_index(drop=True)
rules_table

Association rules with c = 0.5: 26.363 ms
7 rules found


,X,Y,Support of X u Y,Confidence
0,"{704, 825}",{39},1035,0.939
1,"{39, 704}",{825},1035,0.935
2,"{39, 825}",{704},1035,0.872
3,{704},{39},1107,0.617
4,{704},{825},1102,0.614
5,{227},{390},1049,0.577
6,{704},"{39, 825}",1035,0.577


## Results

Results with the default parameters `s = 1%` (support threshold 1,000 baskets) and `c = 0.5` on the T10I4D100K dataset (100,000 baskets):

| Itemset size | Frequent itemsets |
|---|---|
| 1 | 375 |
| 2 | 9 |
| 3 | 1 |

The only frequent triple is `{39, 704, 825}`, with support 1,035. With `c = 0.5` there are 7 association rules:

| Rule | Support of X u Y | Confidence |
|---|---|---|
| {704, 825} -> {39} | 1035 | 0.939 |
| {39, 704} -> {825} | 1035 | 0.935 |
| {39, 825} -> {704} | 1035 | 0.872 |
| {704} -> {39} | 1107 | 0.617 |
| {704} -> {825} | 1102 | 0.614 |
| {227} -> {390} | 1049 | 0.577 |
| {704} -> {39, 825} | 1035 | 0.577 |

**Execution time** (Python 3.11, Windows 11):

| Step | Time |
|---|---|
| Reading the dataset | ~0.3 s |
| A-Priori, frequent itemsets (`s = 1%`) | ~2–3 s |
| Association rules (`c = 0.5`) | ~0.1 ms |

Generating the rules is almost instantaneous because it only uses the support counts already computed by A-Priori and never reads the baskets again. Most of the A-Priori time is spent in the first two passes over the 100,000 baskets.

## AI disclosure

We used Claude through Claude Code in VS Code, during this homework. We used it for:

- Support while programming 
- Checking our code for errors 
- Helping us understand the concepts better
